# 2. Is the extracted text the right document?

We recovered a document for 116,983 of 117,926 filings, which is 99.2%. That
is a recovery rate: it says something came back, not that it was the earnings
release. The report promises an accuracy and this notebook produces it.

The check only means something if the text under the reader's eyes is the
text the project uses. So this notebook does not write a second extractor.
Cell 3 holds the parsing code of `fetch_releases.py`, pasted in unchanged,
and cell 5 verifies that re-running it on a filing gives back the same
exhibit and the same word count that were recorded when the full run was
done. Anything that does not match is reported, not quietly fixed.

Three things can be wrong, and only a reader can see them.

1. **Wrong exhibit.** The filing carries several EX-99 documents and we took
   one that is not the earnings release.
2. **Not an earnings release.** Item 2.02 is also used for monthly traffic
   and sales figures, so the document can be a legitimate exhibit of the
   wrong kind.
3. **Damaged text.** The rule that drops financial-statement tables dropped
   the body instead, or kept a reconciliation appendix whole.

Run it top to bottom. It produces the accuracy rate on its own:
cell 9 makes a first pass with the code, cells 10 to 12 report it.
Then correct the sheet by hand and run cells 10 to 12 again, and
the same sections report the human numbers instead.

In [1]:
# SETUP
# pandas is the only dependency. Pinning an exact version fights the numpy
# Colab ships with and forces a restart, so this installs only if the
# runtime is older than what the code needs.
%pip install -q "pandas>=2.0"

import glob, os, zipfile
import pandas as pd

# The acquisition pass is off. It needs a WRDS account and about a day of
# fetching from EDGAR, and it is what produced the files in nov2_data.
RUN_FROM_SCRATCH = False
WRDS_USER = ""

print("pandas", pd.__version__)


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\jeanm\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


pandas 3.0.5


In [2]:
#where the files are
#
#two folders and one rule.
#  1_2nov/data  the base files, what we pulled from WRDS and the SEC.
#               nothing is ever written here. (formerly nov2_data)
#  files csv    everything the scripts build, including the rebuilt
#               event_table.csv with the EDGAR times read from the filing
#               headers instead of the submissions JSON.
#
#each part has the same layout as parts 3-5: code/, data/, cache/,
#results/ and logs/. paths come from scripts/project_paths.py, so the
#notebook runs from any folder of the checkout.
#
#the rule is in 1_2nov/code/paths.py: a script reads the version in files csv when there
#is one, and the base version otherwise. that is what stops the loop we had,
#where build_event_table.py rewrote the file fetch_acceptance.py had just
#read and nothing downstream could tell which version it got.

import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, NOV2_DATA,
                           PART1_DIR, PART1_CODE, PART2_DIR, PART2_CODE)

# Notebook paths.
CODE = str(PART2_CODE)
DATA = str(PART1_DIR / "data")             # part 1 base files (formerly nov2_data)
BUILT = str(NOV2_DATA)                     # files csv
RESULTS = str(PART2_DIR / "results")
CACHE = str(PART2_DIR / "cache")
LOGS = str(PART2_DIR / "logs")
MANUAL = str(PART2_DIR / "Manual validation")

for d in (CODE, BUILT, RESULTS, CACHE, LOGS, MANUAL):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)

#the scripts read these two names out of the environment
os.environ["NOV2_BASE"] = DATA
os.environ["NOV2_BUILT"] = BUILT

NEED = ["attrition_funnel.csv", "raw_counts.csv", "matched_sample.csv",
        "extraction_report.csv", "timing_validation.csv", "wire_times.csv",
        "three_source_timing.csv", "event_table.csv"]

if not os.path.isdir(DATA):
    raise FileNotFoundError(
        f"no base data at {DATA}. copy the former nov2_data files "
        f"into 1_2nov/data.")

missing = [f for f in NEED
           if not os.path.isfile(os.path.join(DATA, f))
           or os.path.getsize(os.path.join(DATA, f)) == 0]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} of the {len(NEED)} base files are not in {DATA}: "
        + ", ".join(missing) + ". "
        "Drive sends the small files first, so this is usually an upload "
        "still running.")

#which version of each file a script will actually read
print("base  ", DATA)
print("built ", BUILT)
print()
for f in sorted(set(NEED) | set(os.listdir(BUILT))):
    b = os.path.join(BUILT, f)
    if os.path.isfile(b) and os.path.getsize(b) > 0:
        print(f"{os.path.getsize(b):>12,}  {f:<30} built")
    elif os.path.isfile(os.path.join(DATA, f)):
        print(f"{os.path.getsize(os.path.join(DATA, f)):>12,}  {f:<30} base")

print()
print("results ->", RESULTS)
print("manual  ->", MANUAL)
print("code    ->", CODE)

base   C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\1_2nov\nov2_data
built  C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\files csv

  13,867,718  acceptance_times.csv           built
         483  attrition_funnel.csv           built
 105,538,999  event_table.csv                built
   1,836,959  extraction_report.csv          built
   8,721,316  extraction_report_full.csv     built
  12,350,500  matched_sample.csv             built
         550  raw_counts.csv                 base
  12,921,864  three_source_timing.csv        built
  13,914,223  three_source_timing_all.csv    built
       4,887  timing_conflicts_50.csv        built
      20,703  timing_handcheck_50.csv        built
  10,424,706  timing_validation.csv          built
  16,906,801  wire_times.csv                 base

results -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\results\2_results
manual  -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in F

In [3]:
# 2. THE EXTRACTION REPORT
# One row per filing: which exhibit we took, its file name, how many words
# came out and how many tables were dropped.

import re

report = pd.read_csv(f"{DATA}/extraction_report.csv",
                     dtype={"accession": str})
report["accession"] = report.accession.str.strip()
report["year"] = pd.to_datetime(report.filing_date, errors="coerce").dt.year

print(f"{len(report):,} filings")
print(report.source.value_counts().head(8).to_string())

117,926 filings
source
EX-99.1            103553
EX-99                9515
8-K body             1862
EX-99.01             1007
none                  941
EX-99.2               372
EX-99.1 CHARTER       196
EX-99.A               151


In [4]:
# 3. THE PIPELINE'S EXTRACTOR
#
# Copied unchanged from 1_2nov/code/fetch_releases.py (every
# definition below is identical). Do not edit it here: a change made in
# this cell would make the check describe this notebook instead
# of the project.

import html, threading, time, urllib.error, urllib.request

USER_AGENT = "MIT 15.S08 project marcusjean501@gmail.com"
RATE = 9.0          # below the SEC's ceiling of 10 requests a second
WORKERS = 8
MIN_WORDS = 500

ROW_RE = re.compile(r"<tr[^>]*>(.*?)</tr>", re.S | re.I)
CELL_RE = re.compile(r"<td[^>]*>(.*?)</td>", re.S | re.I)
HREF_RE = re.compile(r'href="([^"]+)"', re.I)
SCRIPT_RE = re.compile(r"<(script|style)\b.*?</\1>", re.S | re.I)
TABLE_RE = re.compile(r"<table\b.*?</table>", re.S | re.I)
TAG_RE = re.compile(r"<[^>]+>")
WS_RE = re.compile(r"[ \t\r\f\v]+")
NL_RE = re.compile(r"\n{3,}")


class RateLimiter:
    """One shared gate, so N workers together stay under `rate` per second."""

    def __init__(self, rate: float):
        self.min_gap = 1.0 / rate
        self.lock = threading.Lock()
        self.next_at = 0.0

    def acquire(self) -> None:
        with self.lock:
            now = time.monotonic()
            wait = max(0.0, self.next_at - now)
            self.next_at = max(now, self.next_at) + self.min_gap
        if wait:
            time.sleep(wait)


def fetch(url: str, limiter: RateLimiter, retries: int = 4) -> bytes | None:
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    for attempt in range(retries):
        limiter.acquire()
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 429, 500, 502, 503, 504):
                time.sleep(min(2 ** attempt, 20))
                continue
            return None
        except Exception:
            time.sleep(min(2 ** attempt, 20))
    return None


def _strip(fragment: str) -> str:
    return html.unescape(TAG_RE.sub(" ", fragment)).replace("\xa0", " ")


def to_text(raw: str) -> tuple[str, int]:
    """Drop financial-statement tables, keep layout tables.

    Older EDGAR filings routinely wrap the whole press release in a <table>
    used purely for layout, so removing every table removes the document. We
    therefore judge each table on its content: one that is mostly digits is a
    statement table and goes, because a statement flattened into a word stream
    is noise for the encoder and the numbers come from I/B/E/S anyway. One
    that reads as prose stays.
    """
    body = SCRIPT_RE.sub(" ", raw)

    dropped = 0

    def decide(m: re.Match) -> str:
        nonlocal dropped
        inner = _strip(m.group(0))
        letters = sum(c.isalpha() for c in inner)
        digits = sum(c.isdigit() for c in inner)
        if digits > 0.30 * max(letters + digits, 1):
            dropped += 1
            return " "
        return " " + inner + " "

    body = TABLE_RE.sub(decide, body)
    body = _strip(body)
    body = NL_RE.sub("\n\n", WS_RE.sub(" ", body))
    return body.strip(), dropped


def looks_like_prose(text: str) -> bool:
    words = text.split()
    if len(words) < MIN_WORDS or text.count(". ") < 10:
        return False
    return sum(c.isdigit() for c in text) / max(len(text), 1) < 0.25


def rank(doc_type: str) -> int:
    t = doc_type.upper()
    if t == "EX-99.1":
        return 0
    if t == "EX-99.2":
        return 1
    if t.startswith("EX-99"):
        return 2
    if t == "8-K":
        return 3
    return 9


def parse_index(body: str, base: str) -> list[tuple[str, str]]:
    """Return [(type, absolute url)] for the documents worth trying."""
    out = []
    for row in ROW_RE.findall(body):
        cells = CELL_RE.findall(row)
        if len(cells) < 4:
            continue
        doc_type = re.sub(r"<[^>]+>", " ", cells[3]).strip()
        if rank(doc_type) == 9:
            continue
        m = HREF_RE.search(cells[2])
        if not m:
            continue
        href = html.unescape(m.group(1))
        # EDGAR serves some documents through an "ix?doc=" viewer wrapper.
        if "ix?doc=" in href:
            href = href.split("ix?doc=", 1)[1]
        url = href if href.startswith("http") else \
            "https://www.sec.gov" + (href if href.startswith("/") else "/" + href)
        out.append((doc_type, url))
    out.sort(key=lambda p: rank(p[0]))
    return out


def one_filing(row: dict, limiter: RateLimiter) -> tuple[dict, str | None]:
    acc = row["accession"]
    cik = str(row["cik"]).lstrip("0")
    base = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{acc.replace('-', '')}/")
    rec = {"accession": acc, "cik": cik,
           "filing_date": row.get("filing_date", ""),
           "source": "", "filename": "", "words": 0, "tables_dropped": 0}

    idx = fetch(base + f"{acc}-index.htm", limiter)
    if idx is None:
        rec["source"] = "index fetch failed"
        return rec, None

    for doc_type, url in parse_index(idx.decode("utf-8", "replace"), base):
        body = fetch(url, limiter)
        if body is None:
            continue
        text, n_tables = to_text(body.decode("utf-8", "replace"))
        if looks_like_prose(text):
            rec["source"] = doc_type if doc_type.upper().startswith("EX-99") \
                else "8-K body"
            rec["filename"] = url.rsplit("/", 1)[-1]
            rec["words"] = len(text.split())
            rec["tables_dropped"] = n_tables
            return rec, text

    rec["source"] = "none"
    return rec, None

print("extractor loaded")

extractor loaded


In [5]:
# 4. THE SAMPLE
#
# Two blocks, answering different questions.
#
# The main block is 100 filings, five a year from 2006 to 2025, drawn at
# random within the year. Representative by construction, so the rate it
# produces is the accuracy of the corpus. Nine in ten will be EX-99.1,
# because nine in ten of the corpus is.
#
# The rare block is eight filings of each thin path. Too few to show up in a
# sample of 100, and the ones most likely to be wrong. Judged the same way
# but kept apart: folding them in would make two per cent of the corpus
# carry fourteen per cent of the sample.
#
# The <TYPE> string EDGAR carries is not a clean label. "EX-99.1",
# "EX-99.01", "EX-99.1 PRESS RLSE" and "EX-99.1EARNINGSRELEA" are the same
# path typed four ways, and there are hundreds of spellings. Stratifying on
# the raw string would give almost every filing its own stratum, so the
# strings are folded into four families first.
#
# Filings where nothing came back are in neither block: they are the 943 the
# recovery rate already counts as failures. The question here is whether the
# documents we did get are the right ones.

SEED = 20261104
PER_YEAR = 5
RARE_EACH = 8
MISSED = {"none", "index fetch failed"}


def family(s):
    s = str(s).strip().upper()
    if s == "8-K BODY":
        return "8-K body"
    m = re.match(r"EX-99[.,`-]*0*(\d+)", s)
    if m:
        return "EX-99.1" if int(m.group(1)) == 1 else "EX-99.n"
    return "EX-99 other"


pool = report[report.year.between(2006, 2025)
              & ~report.source.isin(MISSED)].copy()
pool["family"] = pool.source.map(family)
print(pool.family.value_counts().to_string(), "\n")

main = (pool.groupby("year", group_keys=False)
            .sample(n=PER_YEAR, random_state=SEED)
            .assign(stratum="main"))

thin = pool.family.value_counts(normalize=True)
thin = thin[thin < 0.02].index
rare = pd.concat(
    [pool[pool.family == f].sample(min(RARE_EACH, (pool.family == f).sum()),
                                   random_state=SEED)
     for f in thin]).assign(stratum="rare")

sample = (pd.concat([main, rare]).drop_duplicates("accession")
            .sort_values(["stratum", "year", "accession"])
            .reset_index(drop=True))

assert (sample.stratum == "main").sum() == 20 * PER_YEAR
print(f"{len(sample)} filings: {(sample.stratum == 'main').sum()} main, "
      f"{(sample.stratum == 'rare').sum()} rare")
print(sample.groupby(["stratum", "family"]).size().to_string())

family
EX-99.1        104844
EX-99 other      9793
8-K body         1862
EX-99.n           484 



116 filings: 100 main, 16 rare
stratum  family     
main     8-K body        1
         EX-99 other     9
         EX-99.1        90
rare     8-K body        8
         EX-99.n         8


In [6]:
# 5. RE-EXTRACT, AND CHECK AGAINST WHAT THE FULL RUN RECORDED
#
# Same code, same filing, so the same exhibit and the same word count should
# come back. A mismatch means EDGAR changed the filing or the recorded row is
# wrong. Either way it is a finding and it gets printed.
#
# About four minutes: two requests a filing, under the SEC's ten a second.

limiter = RateLimiter(RATE)

# Keep what came over the wire, so section 6 can re-run the table rule
# without fetching anything twice. The extraction path is untouched:
# one_filing still calls fetch, fetch still does the fetching, it just
# remembers the bytes on the way past.
FETCHED = {}
_fetch = fetch


def fetch(url, limiter, retries=4):
    body = _fetch(url, limiter, retries)
    if body is not None:
        FETCHED[url] = body
    return body


def to_text_keep_tables(raw):
    """to_text with the table rule removed, and nothing else changed."""
    body = _strip(SCRIPT_RE.sub(" ", raw))
    return NL_RE.sub("\n\n", WS_RE.sub(" ", body)).strip(), 0


texts, raw_html, checks = {}, {}, []
for i, row in sample.iterrows():
    rec, text = one_filing({"accession": row.accession, "cik": row.cik,
                            "filing_date": row.filing_date}, limiter)
    texts[row.accession] = text or ""
    if rec["filename"]:
        for url, body in FETCHED.items():
            if url.endswith("/" + rec["filename"]):
                raw_html[row.accession] = body.decode("utf-8", "replace")
                break
    checks.append({
        "accession": row.accession,
        "source_recorded": row.source, "source_now": rec["source"],
        "words_recorded": int(row.words), "words_now": int(rec["words"]),
        "same_exhibit": rec["source"] == row.source,
        "same_words": int(rec["words"]) == int(row.words),
    })
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(sample)}", flush=True)

chk = pd.DataFrame(checks)
agree = chk.same_exhibit & chk.same_words
print(f"\nreproduces the recorded extraction: {agree.sum()}/{len(chk)}")
if (~agree).any():
    print("\ndoes not reproduce:")
    print(chk[~agree].to_string(index=False))
chk.to_csv(os.path.join(RESULTS, "extraction_reproduction.csv"), index=False)

  20/116


  40/116


  60/116


  80/116


  100/116



reproduces the recorded extraction: 116/116


In [7]:
# 6. WHAT DROPPING THE FINANCIAL TABLES TAKES AWAY
#
# The extractor keeps a table that reads as prose and drops one that is
# mostly digits, because a statement flattened into a word stream is noise
# for the encoder and the figures come from I/B/E/S anyway. That is a
# judgement, so it has to be measured rather than asserted: this re-extracts
# the same filings with the rule turned off and compares.
#
# Two things matter. How much narrative went out with the tables, which
# should be near zero and is a bug when it is not. And how many numbers went
# out, which should be large and is the point of the rule.

NUM_RE = re.compile(r"\b\d[\d,.]*%?\b")

loss = []
for acc, kept in texts.items():
    raw = raw_html.get(acc)
    if not raw or not kept:
        continue
    whole, _ = to_text_keep_tables(raw)
    w_whole, w_kept = whole.split(), kept.split()
    n_whole = len(NUM_RE.findall(whole))
    n_kept = len(NUM_RE.findall(kept))
    # Words that are not numbers: the narrative proper.
    prose_whole = sum(1 for w in w_whole if not NUM_RE.fullmatch(w))
    prose_kept = sum(1 for w in w_kept if not NUM_RE.fullmatch(w))
    loss.append({
        "accession": acc,
        "words_with_tables": len(w_whole), "words_kept": len(w_kept),
        "numbers_with_tables": n_whole, "numbers_kept": n_kept,
        "prose_with_tables": prose_whole, "prose_kept": prose_kept,
        "pct_numbers_dropped": round(
            100 * (n_whole - n_kept) / max(n_whole, 1), 1),
        "pct_prose_dropped": round(
            100 * (prose_whole - prose_kept) / max(prose_whole, 1), 1),
    })

ls = pd.DataFrame(loss)
print(f"{len(ls)} filings compared with and without the table rule\n")
print(f"numbers removed   median {ls.pct_numbers_dropped.median():5.1f}%  "
      f"mean {ls.pct_numbers_dropped.mean():5.1f}%")
print(f"narrative removed median {ls.pct_prose_dropped.median():5.1f}%  "
      f"mean {ls.pct_prose_dropped.mean():5.1f}%")

# A filing that lost a lot of narrative is the failure mode this rule has:
# the press release was itself laid out as a table and went out whole.
suspect = ls[ls.pct_prose_dropped > 40].sort_values(
    "pct_prose_dropped", ascending=False)
print(f"\nlost more than 40% of their narrative: {len(suspect)}")
if len(suspect):
    print(suspect[["accession", "pct_prose_dropped", "words_kept"]]
          .head(10).to_string(index=False))
    print("\nThese are the ones to read first in the sheet below: if the")
    print("rule ate the release, text_usable is n and the fix is in to_text.")

ls.to_csv(os.path.join(RESULTS, "table_rule_loss.csv"), index=False)
print("\n->", os.path.join(RESULTS, "table_rule_loss.csv"))

116 filings compared with and without the table rule

numbers removed   median  38.5%  mean  26.4%
narrative removed median  11.2%  mean   6.2%

lost more than 40% of their narrative: 7
           accession  pct_prose_dropped  words_kept
0001171843-13-002153               67.1        1541
0000950153-06-000420               65.1        1182
0001558370-21-009071               55.5        5300
0001558370-23-019292               53.2        4173
0001043509-20-000020               52.5        1399
0001030469-25-000033               48.6        2932
0001558370-19-003322               42.0        1562

These are the ones to read first in the sheet below: if the
rule ate the release, text_usable is n and the fix is in to_text.

-> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\results\2_results\table_rule_loss.csv


In [8]:
# 7. THE SHEET, AND THE VIEWER
#
# One row per filing: a link to the filing on EDGAR and the text our pipeline
# produced for it. Three judgements and a note. The viewer is the same thing
# as one page, which is faster to work through than a spreadsheet.

import html

SHEET = os.path.join(MANUAL, "extraction_handcheck.csv")


def edgar(cik, acc):
    return (f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/"
            f"{acc.replace('-', '')}/{acc}-index.htm")


sheet = pd.DataFrame({
    "n": range(1, len(sample) + 1),
    "stratum": sample.stratum.values,
    "year": sample.year.values,
    "accession": sample.accession.values,
    "exhibit_taken": sample.source.values,
    "family": sample.family.values,
    "filename": sample.filename.values,
    "words": sample.words.values,
    "tables_dropped": sample.tables_dropped.values,
    "edgar_index": [edgar(c, a) for c, a in zip(sample.cik, sample.accession)],
    "right_exhibit": "",
    "is_earnings_release": "",
    "text_usable": "",
    "note": "",
})
sheet.to_csv(SHEET, index=False, encoding="utf-8-sig")

parts = ["<html><head><meta charset='utf-8'><style>"
         "body{font-family:Georgia,serif;margin:40px;max-width:900px}"
         "section{border-top:1px solid #ccc;padding-top:20px;margin-top:40px}"
         "pre{white-space:pre-wrap;background:#f6f6f6;padding:16px;"
         "max-height:520px;overflow:auto;font-family:monospace;font-size:13px}"
         "</style></head><body><h1>Extraction hand check</h1>"
         "<p>For each filing: open it on EDGAR, read what we extracted, and "
         "answer the three questions in the sheet.</p>"]
for _, r in sheet.iterrows():
    t = texts.get(r.accession, "")
    parts.append(
        f"<section><h2>{r.n}. {r.accession} &middot; {r.year} "
        f"&middot; {r.stratum}</h2>"
        f"<p>we took <b>{html.escape(str(r.exhibit_taken))}</b> "
        f"({html.escape(str(r.filename))}), {r.words:,} words, "
        f"{r.tables_dropped} tables dropped &middot; "
        f"<a href='{r.edgar_index}' target='_blank'>open on EDGAR</a></p>"
        f"<pre>{html.escape(t) if t else 'NOTHING EXTRACTED'}</pre></section>")
parts.append("</body></html>")

page = os.path.join(MANUAL, "extraction_handcheck.html")
open(page, "w", encoding="utf-8").write("\n".join(parts))

print("sheet  ->", SHEET)
print("viewer ->", page)

sheet  -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\2_EDGAR text\Manual validation\extraction_handcheck.csv
viewer -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\2_EDGAR text\Manual validation\extraction_handcheck.html


In [9]:
# 8. THE CLEANED TEXTS, AS A DATASET
#
# One row per filing: the accession number, the timestamps, the exhibit we
# took and the text itself. Keyed on the accession, so it joins to
# matched_sample.csv, to the event table and to the surprise file without
# anybody having to guess a key. That is the thing an earlier copy of
# nov2_data got wrong, and it is why nothing in it could be joined.
#
# This covers the filings this notebook fetched. The full corpus of 116,983
# texts is produced by fetch_releases.py in notebook 1, one zip a year under
# 1_2nov/data/releases_text/, with the same accession naming.

# Timestamps come from the rebuilt event table in files csv (EDGAR times read
# from the filing headers), not from the base table in 1_2nov/data, which still
# carries the submissions-JSON times and only serves to build the data in part 1.
cols = ["accession", "acceptance_et", "session_date", "when"]
head = pd.read_csv(f"{BUILT}/event_table.csv", nrows=0).columns
if "release_et" in head:
    cols.insert(2, "release_et")
ev = pd.read_csv(f"{BUILT}/event_table.csv", dtype={"accession": str},
                 usecols=cols).drop_duplicates("accession")

ds = pd.DataFrame({"accession": list(texts),
                   "text": [texts[a] for a in texts]})
ds["words"] = ds.text.str.split().str.len()
ds = (ds.merge(sample[["accession", "source", "filename", "family",
                       "tables_dropped"]], on="accession", how="left")
        .merge(ev, on="accession", how="left"))
ds = ds[[c for c in
         ["accession", "acceptance_et", "release_et", "session_date", "when",
          "source", "family", "filename", "tables_dropped", "words", "text"]
         if c in ds.columns]]

path = os.path.join(RESULTS, "release_texts.csv")
ds.to_csv(path, index=False, encoding="utf-8")
print(f"{len(ds):,} texts, {ds.words.sum():,} words")
print(f"with a timestamp: {ds.acceptance_et.notna().sum():,}")
print("columns:", ", ".join(ds.columns))
print("->", path)

116 texts, 407,038 words
with a timestamp: 116
columns: accession, acceptance_et, release_et, session_date, when, source, family, filename, tables_dropped, words, text
-> /Users/mahautbonnetderoovere/Desktop/Github/S08/2_EDGAR text/results/release_texts.csv


In [10]:
# 9. A FIRST PASS, MADE BY THE CODE
#
# The brief asks for an accuracy rate. A rate that only exists once somebody
# has read 116 filings is a rate the notebook cannot produce, so the code
# makes its own pass first and the reader corrects it. Three questions,
# three tests, none of which needs judgement:
#
#   right_exhibit        refetch every EX-99 document in the filing and
#                        check that the one we took scores highest on
#                        earnings language. If another one beats it, we
#                        took the wrong exhibit.
#   is_earnings_release  earnings vocabulary present, and the vocabulary of
#                        a monthly traffic or sales release absent. Item
#                        2.02 carries both kinds and they read differently.
#   text_usable          the narrative survived the table rule, and what is
#                        left is long enough to be a release.
#
# This is a machine pass and it is labelled as one. It is written into the
# sheet so the reader starts from a filled form and changes what is wrong,
# which is faster and more honest than starting from blanks, and section 10
# reports the two separately.

EARNINGS = ["earnings per share", "net income", "diluted", "fourth quarter",
            "first quarter", "second quarter", "third quarter", "full year",
            "results of operations", "financial results", "net revenue",
            "operating income", "gaap", "guidance"]
NOT_EARNINGS = ["traffic", "revenue passenger", "load factor",
                "available seat miles", "same-store sales for",
                "monthly sales", "sales for the month", "unit sales for"]


def score_text(t):
    low = t[:8000].lower()
    return (sum(k in low for k in EARNINGS),
            sum(k in low for k in NOT_EARNINGS))


auto = []
for _, row in sample.iterrows():
    acc, text = row.accession, texts.get(row.accession, "")
    if not text:
        auto.append({"accession": acc, "right_exhibit_auto": "n",
                     "is_earnings_release_auto": "n",
                     "text_usable_auto": "n",
                     "auto_note": "nothing extracted"})
        continue

    hits, anti = score_text(text)

    # Every EX-99 in the same filing, scored the same way, so "did we take
    # the right one" is answered against the alternatives and not in the
    # abstract.
    cik = str(int(row.cik))
    base = (f"https://www.sec.gov/Archives/edgar/data/{cik}/"
            f"{acc.replace('-', '')}/")
    best_other, best_name = -1, ""
    idx = fetch(base + f"{acc}-index.htm", limiter)
    if idx is not None:
        for doc_type, url in parse_index(idx.decode("utf-8", "replace"),
                                         base):
            name = url.rsplit("/", 1)[-1]
            if name == row.filename:
                continue
            body = fetch(url, limiter)
            if body is None:
                continue
            other, _ = to_text(body.decode("utf-8", "replace"))
            if not looks_like_prose(other):
                continue
            h, a = score_text(other)
            if h - a > best_other:
                best_other, best_name = h - a, name

    lost = ls.loc[ls.accession == acc, "pct_prose_dropped"]
    lost = float(lost.iloc[0]) if len(lost) else 0.0

    right = "y" if (hits - anti) >= best_other else "n"
    release = "y" if (hits >= 3 and anti <= 1) else "n"
    usable = "y" if (lost <= 40 and len(text.split()) >= 300) else "n"

    note = []
    if right == "n":
        note.append(f"{best_name} scores higher")
    if release == "n":
        note.append(f"earnings words {hits}, traffic words {anti}")
    if usable == "n":
        note.append(f"{lost:.0f}% of the narrative dropped, "
                    f"{len(text.split()):,} words left")
    auto.append({"accession": acc, "right_exhibit_auto": right,
                 "is_earnings_release_auto": release,
                 "text_usable_auto": usable,
                 "auto_note": "; ".join(note)})

au = pd.DataFrame(auto)
for c in ["right_exhibit", "is_earnings_release", "text_usable"]:
    print(f"{c:<22} n = {(au[c + '_auto'] == 'n').sum():>3}")

# Fill the sheet with the machine's answers. A reader overwrites a cell to
# disagree; the _auto columns keep what the code thought, so section 10 can
# say how often the two differ.
sheet2 = pd.read_csv(SHEET).merge(au, on="accession", how="left")
for c in ["right_exhibit", "is_earnings_release", "text_usable"]:
    sheet2[c] = sheet2[c + "_auto"]
sheet2["judged_by"] = "code"
sheet2.to_csv(SHEET, index=False, encoding="utf-8-sig")
print("\nsheet refilled with the machine pass ->", SHEET)

right_exhibit          n =   7
is_earnings_release    n =   6
text_usable            n =   7

sheet refilled with the machine pass -> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\2_EDGAR text\Manual validation\extraction_handcheck.csv


## Fill the sheet now

Open `extraction_handcheck.html` from the **Manual validation** folder, work
down it with the sheet beside you, and answer three questions per filing with
`y` or `n`.

`right_exhibit` — is the document we took the earnings release, rather than
another exhibit of the same filing?

`is_earnings_release` — is it an earnings release at all? A monthly traffic
or sales release filed under Item 2.02 is a legitimate exhibit and the wrong
document for us.

`text_usable` — is the narrative intact? The extractor drops tables that are
mostly digits. It is wrong if it took the body away with them, and it is
wrong if it left a reconciliation appendix whole.

Put anything odd in `note`, in your own words. The notes are what the report
quotes; the rate on its own says nothing about what went wrong.

In [11]:
# 10. THE ACCURACY RATE
#
# Run this once the sheet is filled. A filing counts as correct only if all
# three answers are yes. The headline rate is the main block alone; the rare
# exhibit types are reported beside it and never folded in.

filled = pd.read_csv(SHEET)
cols = ["right_exhibit", "is_earnings_release", "text_usable"]


def yes(s):
    return s.astype(str).str.strip().str.lower().isin(["y", "yes", "true", "1"])


def no(s):
    return s.astype(str).str.strip().str.lower().isin(["n", "no", "false", "0"])


answered = filled[cols].apply(yes) | filled[cols].apply(no)
done = filled[answered.all(axis=1)]

# Where a reader has overwritten the machine's answer, the row is theirs.
auto_cols = [c + "_auto" for c in cols]
if all(c in filled.columns for c in auto_cols):
    changed = (filled[cols].values != filled[auto_cols].values).any(axis=1)
    n_human = int(changed.sum())
    print(f"judged: {len(done)}/{len(filled)}   "
          f"{n_human} rows corrected by hand, "
          f"{len(filled) - n_human} as the code left them")
    if not n_human:
        print("This is the machine pass. The rate below is provisional "
              "until the sheet is read.")
else:
    print(f"judged: {len(done)}/{len(filled)}")

# Not an error: the sheet is filled by hand between cell 6 and this one, so
# on the first pass there is nothing to count yet.
lines = []
if len(done):
    done = done.assign(correct=done[cols].apply(yes).all(axis=1))

for block in (["main", "rare"] if len(done) else []):
    b = done[done.stratum == block]
    if not len(b):
        continue
    label = ("extraction accuracy" if block == "main"
             else "rare exhibit types, reported separately")
    print(f"\n{label}: {100 * b.correct.mean():.1f}%  "
          f"({b.correct.sum()}/{len(b)})")
    lines.append({"block": block, "n": len(b), "correct": int(b.correct.sum()),
                  "accuracy_pct": round(100 * b.correct.mean(), 1)})
    for c in cols:
        bad = no(b[c])
        print(f"  fails {c:<22} {bad.sum():>3}  {100 * bad.mean():5.1f}%")

if not len(done):
    print("\nFill extraction_handcheck.csv in the Manual validation folder,")
    print("then run this cell again. Three columns a row, y or n.")
else:
    print("\nby exhibit family:")
    print(done.groupby(["stratum", "family"]).correct
              .agg(["size", "mean"]).to_string())
    print("\nnotes on the failures:")
    for _, r in done[~done.correct].iterrows():
        why = r.get("note")
        if not isinstance(why, str) or not why.strip():
            why = r.get("auto_note", "")
        print(f"  {r.stratum:<5} {r.accession}  {r.exhibit_taken:<10} {why}")

    pd.DataFrame(lines).to_csv(
        os.path.join(RESULTS, "extraction_accuracy.csv"), index=False)
    done.to_csv(os.path.join(RESULTS, "extraction_handcheck_judged.csv"),
                index=False)
    print("\n->", os.path.join(RESULTS, "extraction_accuracy.csv"))

judged: 116/116   0 rows corrected by hand, 116 as the code left them
This is the machine pass. The rate below is provisional until the sheet is read.

extraction accuracy: 85.0%  (85/100)
  fails right_exhibit            5    5.0%
  fails is_earnings_release      3    3.0%
  fails text_usable              7    7.0%

rare exhibit types, reported separately: 81.2%  (13/16)
  fails right_exhibit            2   12.5%
  fails is_earnings_release      3   18.8%
  fails text_usable              0    0.0%

by exhibit family:
                     size      mean
stratum family                     
main    8-K body        1  0.000000
        EX-99 other     9  0.666667
        EX-99.1        90  0.877778
rare    8-K body        8  0.875000
        EX-99.n         8  0.750000

notes on the failures:
  main  0000950153-06-000420  EX-99.1    65% of the narrative dropped, 1,182 words left
  main  0001299933-06-005371  8-K body   earnings words 1, traffic words 0
  main  0001104659-12-083422  EX-99  

In [12]:
# 11. WHAT TO CHANGE IN THE EXTRACTOR
#
# The brief says to improve the extraction code if we find systematic
# errors. A rate alone cannot say that: three failures are three accidents,
# three failures that are all the same shape are a rule to change. This
# sorts the judged failures by which part of fetch_releases.py decides the
# outcome, so the answer is a function name and not an impression.
#
#   rank() and parse_index()  choose which exhibit to try, in order
#   looks_like_prose()        decides whether a document is accepted at all
#   to_text()                 decides which tables survive

# Part one runs whether or not anybody has filled the sheet, because these
# three tests need no judgement: they are the extractor caught contradicting
# itself. Part two needs the sheet, and says so.

print("WHAT THE CODE CAN SEE WITHOUT A READER\n")

ate = ls[ls.pct_prose_dropped > 40]
print(f"{len(ate):>3}  the table rule removed more than 40% of the "
      f"narrative")
print("     to_text judges a table by its digit ratio and drops it above "
      "0.30.")
print("     A release laid out as one table goes out whole. Of the three")
print("     failure modes this is the only one the code can prove alone.")

short = sample[sample.words < 300]
print(f"\n{len(short):>3}  fewer than 300 words kept")
print("     looks_like_prose accepts at 500 words before the table rule")
print("     runs, so anything this short lost its body afterwards.")

multi = report[report.accession.isin(sample.accession)]
odd = multi[~multi.source.astype(str).str.upper().str.match(r"EX-99[.,`-]*0*1")]
print(f"\n{len(odd):>3}  an exhibit other than EX-99.1 was taken")
print("     rank() tries EX-99.1 first, so these are filings where it was")
print("     absent or failed looks_like_prose. Not wrong, but the place")
print("     where taking the wrong document is most likely.")

flagged = sorted(set(ate.accession) | set(short.accession)
                 | set(odd.accession))
print(f"\n{len(flagged)} filings flagged by at least one of the three. "
      f"Read these first.")
pd.DataFrame({"accession": flagged}).to_csv(
    os.path.join(RESULTS, "extraction_flagged.csv"), index=False)

print("\n" + "-" * 68)
print("WHAT NEEDS THE SHEET\n")

if not len(done) or done.correct.all():
    print("Nothing judged yet, or nothing judged has failed. The three")
    print("counts above stand on their own; the split below needs the")
    print("sheet, because only a reader can say whether the document is")
    print("the earnings release.")
else:
    bad = done[~done.correct].merge(
        ls[["accession", "pct_prose_dropped", "pct_numbers_dropped"]],
        on="accession", how="left")

    buckets = {
        "rank / parse_index: took the wrong exhibit":
            no(bad.right_exhibit),
        "looks_like_prose: accepted a document that is not a release":
            no(bad.is_earnings_release),
        "to_text: the table rule damaged the text":
            no(bad.text_usable),
    }
    for label, mask in buckets.items():
        k = int(mask.sum())
        share = 100 * k / max(len(done), 1)
        verdict = ("systematic, worth changing the rule" if share >= 3
                   else "too few to call systematic")
        print(f"{k:>3}  {share:4.1f}% of the sample   {label}")
        print(f"     {verdict}")

    ate = bad[(bad.pct_prose_dropped > 40) & no(bad.text_usable)]
    if len(ate):
        print(f"\n{len(ate)} of the damaged texts also lost more than 40% of")
        print("their narrative to the table rule. That is one cause, not two:")
        print("the release was laid out as a table and to_text dropped it on")
        print("its digit ratio. The threshold is the 0.30 in to_text.")

    bad.to_csv(os.path.join(RESULTS, "extraction_failures.csv"), index=False)
    print("\n->", os.path.join(RESULTS, "extraction_failures.csv"))

WHAT THE CODE CAN SEE WITHOUT A READER

  7  the table rule removed more than 40% of the narrative
     to_text judges a table by its digit ratio and drops it above 0.30.
     A release laid out as one table goes out whole. Of the three
     failure modes this is the only one the code can prove alone.

  0  fewer than 300 words kept
     looks_like_prose accepts at 500 words before the table rule
     runs, so anything this short lost its body afterwards.

 25  an exhibit other than EX-99.1 was taken
     rank() tries EX-99.1 first, so these are filings where it was
     absent or failed looks_like_prose. Not wrong, but the place
     where taking the wrong document is most likely.

30 filings flagged by at least one of the three. Read these first.

--------------------------------------------------------------------
WHAT NEEDS THE SHEET

  7   6.0% of the sample   rank / parse_index: took the wrong exhibit
     systematic, worth changing the rule
  6   5.2% of the sample   looks_like_

In [13]:
# 12. THE FIGURES
# Three charts into results/, because a csv of rates is not something
# anybody reads twice.

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))

v = report.source.map(family).value_counts()
ax[0].barh(v.index[::-1], v.values[::-1], color="#4a5568")
ax[0].set_title("Which exhibit the extractor took")
ax[0].set_xlabel("filings")
for i, n in enumerate(v.values[::-1]):
    ax[0].text(n, i, f" {n:,}", va="center", fontsize=9)

ax[1].hist(ls.pct_numbers_dropped, bins=30, color="#4a5568")
ax[1].set_title("Numbers removed with the tables")
ax[1].set_xlabel("% of numeric tokens dropped")
ax[1].set_ylabel("filings")

ax[2].hist(ls.pct_prose_dropped, bins=30, color="#9b2c2c")
ax[2].axvline(40, color="black", lw=1, ls="--")
ax[2].set_title("Narrative removed with the tables")
ax[2].set_xlabel("% of non-numeric words dropped")

for a in ax:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
p1 = os.path.join(RESULTS, "extraction_quality.png")
fig.savefig(p1, dpi=150)
print("->", p1)

if len(done) and "correct" in done:
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    g = done.groupby("family").correct.mean().sort_values() * 100
    ax2.barh(g.index, g.values, color="#2c5282")
    ax2.set_xlim(0, 100)
    ax2.set_xlabel("% correct")
    ax2.set_title("Extraction accuracy by exhibit family")
    for i, n in enumerate(g.values):
        ax2.text(n, i, f" {n:.0f}%", va="center", fontsize=9)
    ax2.spines[["top", "right"]].set_visible(False)
    fig2.tight_layout()
    p2 = os.path.join(RESULTS, "extraction_accuracy.png")
    fig2.savefig(p2, dpi=150)
    print("->", p2)
else:
    print("accuracy chart skipped: the sheet is not filled yet")

-> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\results\2_results\extraction_quality.png
-> C:\Users\jeanm\Downloads\15.S08 NLP and LLMs in Finance\Rendu Mahaut\results\2_results\extraction_accuracy.png
